# Image Retrieval

- This notebook contains code to carry out image retrieval based on the CLIP Image Embeddings generated for the LAION-5B dataset.

## Non-Faiss Exact Cosine Similarity Image Search

This function searches a dataset of precomputed CLIP embeddings to find images most similar to a given text prompt, an image, or a combination of both. It first computes normalized embeddings for the query using the specified CLIP model, then iterates through the dataset stored in Parquet files, loading embeddings in manageable chunks. For each chunk, it calculates cosine similarities between the query and dataset embeddings, combines text and image similarities using a weighted alpha parameter, filters results based on an optional similarity threshold, and collects metadata such as URLs, captions, and original image indices. Finally, it sorts all matches by similarity and returns the top results if requested.

In [1]:
from transformers import AutoProcessor, AutoModel
from PIL import Image
import torch
from typing import List, Optional
import numpy as np
import pyarrow.dataset as ds
import os
import re
import pyarrow.parquet as pq

def find_similar_images(
    dataset_dir: str,
    model_name: str,
    text_prompt: Optional[str] = None,
    image_path: Optional[str] = None,
    top_n: Optional[int] = None,
    similarity_threshold: Optional[float] = None,
    alpha: float = 0.5
) -> List[dict]:
    ''' 
    Functions used to find images similar to a text or image (or both) based on the CLIP embeddings.

    Inputs:
    - dataset_dir: Directory containing the CLIP embeddings dataset in Parquet format.
    - model_name: Name of the pre-trained CLIP model to use.
    - text_prompt: Optional text prompt to find similar images. (None means only the image is used to search).
    - image_path: Optional path to an image to find similar images. (None means only the text prompt is used to search).
    - top_n: Optional number of top similar images to return. (None means return all).
    - similarity_threshold: Optional threshold for cosine similarity to filter results. (None means no filtering).
    - alpha: Weight for text similarity in the combined similarity score. (0.5 - equal weight between text and image, 1 - text prompt only is used, 0 - image only is used).

    Outputs:
    - List of dictionaries containing URLs, captions, original image indices, and cosine similarities of the most similar images.
    '''

    assert text_prompt or image_path, "You must provide either a text prompt or an image path."

    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    text_embedding = None
    image_embedding = None

    if text_prompt:
        inputs = processor(text=text_prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        text_embedding = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    if image_path:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    # Convert to numpy for similarity calculation
    text_embedding_np = text_embedding.cpu().squeeze().numpy() if text_embedding is not None else None
    image_embedding_np = image_embedding.cpu().squeeze().numpy() if image_embedding is not None else None

    all_matches = []

    # Loads only matching files from dataset_dir like part-00000.parquet to avoid loading checkpoints
    pattern = re.compile(r"^part-\d{5}\.parquet$")
    valid_files = [
        os.path.join(dataset_dir, f)
        for f in os.listdir(dataset_dir)
        if pattern.match(f)
    ]
    dataset = ds.dataset(valid_files, format="parquet")

    # Iterate through each fragment (.parquet file) composing the dataset
    for fragment in dataset.get_fragments():
        print(f"Processing fragment: {fragment.path}")
        # pq_file = pq.ParquetFile(os.path.join(dataset_dir, fragment.path))
        pq_file = pq.ParquetFile(fragment.path)

        for row_group_index in range(pq_file.num_row_groups):
            table = pq_file.read_row_group(row_group_index)
            df_chunk = table.to_pandas()

            df_chunk = df_chunk.dropna(subset=['embeddings_result'])
            if df_chunk.empty:
                continue

            df_chunk['embeddings_result'] = df_chunk['embeddings_result'].apply(
                lambda x: np.array(x) if isinstance(x, list) else x
            )
            image_embeddings = np.vstack(df_chunk['embeddings_result'].values)

            # Compute similarities separately
            sim_text = np.dot(image_embeddings, text_embedding_np.T) if text_embedding_np is not None else 0
            sim_image = np.dot(image_embeddings, image_embedding_np.T) if image_embedding_np is not None else 0

            # Combine similarity with weights
            combined_sim = None
            if text_embedding_np is not None and image_embedding_np is not None:
                combined_sim = alpha * sim_text + (1 - alpha) * sim_image
            elif text_embedding_np is not None:
                combined_sim = sim_text
            else:
                combined_sim = sim_image

            df_chunk['combined_similarity'] = combined_sim

            if similarity_threshold is not None:
                df_chunk = df_chunk[df_chunk['combined_similarity'] >= similarity_threshold]

            for _, row in df_chunk.iterrows():
                all_matches.append({
                    'url': row.get('url'),
                    'caption': row.get('caption'),
                    'original_image_index': row.get('original_image_index'),
                    'cosine_similarity': row['combined_similarity']
                })

    all_matches.sort(key=lambda x: x['cosine_similarity'], reverse=True)
    top_matches = all_matches[:top_n] if top_n is not None else all_matches

    return top_matches

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\2_ImageRetrieval\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
dataset_dir = r'..\clip_embeddings_resumable_symlink\all_images_openai_clip_vit_large_patch14\0000_embeddings'
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_name = clip_model_names[2] 
text_prompt = "umpire"
search_image_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images\test\TestConstructionWorkerImage.jpg' 
top_n = 10
similarity_threshold = 0.15

image_info_list = find_similar_images(
    dataset_dir=dataset_dir,
    model_name=model_name,
    text_prompt=text_prompt,
    image_path = search_image_path,
    top_n=top_n,
    similarity_threshold=similarity_threshold,
    alpha=1  # 1 = text only, 0 = image only, 0.5 = equal weighting
)

Processing fragment: ../clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00000.parquet
Processing fragment: ../clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00001.parquet
Processing fragment: ../clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00002.parquet
Processing fragment: ../clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00003.parquet
Processing fragment: ../clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00004.parquet
Processing fragment: ../clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00005.parquet
Processing fragment: ../clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00006.parquet
Processing fragment: ../clip_embeddings_resumable_symlink/all_

## Faiss Exact Cosine Similarity Image Search

* **`build_faiss_index_with_mapping_resume_exact` function:**
  This function builds or resumes a FAISS index from a list of Parquet files containing precomputed embeddings. It reads embeddings in batches, normalizes them for cosine similarity, and adds them to an exact `IndexFlatIP` FAISS index. The function supports resuming from an existing index and mapping, skipping already indexed embeddings. It maintains a mapping between vector indices and image paths, and periodically saves both the FAISS index and mapping to disk to prevent data loss. It processes large datasets efficiently by iterating over batches and shards, ensuring that even very large embeddings can be indexed without exceeding memory limits.

* **`process_all_prompts_with_resume` function:**
  This function performs an exact similarity search across multiple FAISS shard indexes, derived from the prior function. It computes normalized embeddings for a given text prompt and/or image using a CLIP model, optionally combining them with a weighted alpha parameter. The function then streams through each FAISS shard, performing inner-product searches to retrieve the most similar images, applying an optional similarity threshold. Results from all shards are aggregated, sorted by similarity, and truncated to the top-k matches if requested. This approach allows searching very large embedding collections without loading all data into memory at once.

In [1]:
import os
import json
import gc
from pathlib import Path

import faiss
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm import tqdm
import pickle
import ast

# Disable tqdm's background monitor thread to reduce flicker in some environments
tqdm.monitor_interval = 0


def build_faiss_index_with_mapping_resume_exact(
    parquet_files,
    index_path,
    mapping_path,
    save_every_vectors: int = 5_000_000,
    state_path=None,
):
    """
    EXACT FAISS IndexFlatIP builder using CPU with resume + checkpointing.

    - Uses inner product (IP) with L2-normalized vectors => cosine similarity.
    - Supports interruption & resume via JSON state file.
    - Periodically checkpoints FAISS index, mapping, and state using atomic writes.
    - Progress reporting via tqdm (per-parquet-file progress in ROW GROUPS).

    Notes
    -----
    - We process each Parquet file row-group by row-group.
    - Resume is tracked per file in units of row groups completed.
    """

    index_path = str(index_path)
    mapping_path = str(mapping_path)

    if state_path is None:
        state_path = mapping_path + ".state.json"
    state_path = str(state_path)

    parquet_files = [str(f) for f in parquet_files]

    # === Step 1: Load existing FAISS index + mapping (CPU) ===
    if os.path.exists(index_path) and os.path.exists(mapping_path):
        try:
            index = faiss.read_index(index_path)
            with open(mapping_path, "rb") as f:
                idx_to_path = pickle.load(f)
            if not isinstance(idx_to_path, list):
                raise ValueError("mapping_path must contain a list.")
            total_vectors_existing = len(idx_to_path)
        except Exception:
            index = None
            idx_to_path = []
            total_vectors_existing = 0
    else:
        index = None
        idx_to_path = []
        total_vectors_existing = 0

    # === Step 2: Load state (per-file groups processed) ===
    if os.path.exists(state_path):
        with open(state_path, "r", encoding="utf-8") as f:
            state = json.load(f)
        # New key: per_file_groups_processed
        per_file_groups_processed = state.get("per_file_groups_processed", {})
        # Old key: per_file_rows_processed (ignored now for safety)
        # If you really want to attempt to map rows->groups, you'd need per-file metadata.
    else:
        per_file_groups_processed = {}

    # -------------------------------------------
    # Atomic write helper
    # -------------------------------------------
    def atomic_write_bytes(filepath: str, write_fn):
        """
        Atomically write to a file by writing to a temp file, (optionally) fsyncing, and renaming.
        `write_fn(tmp_path)` should fully write the file.
        """
        tmp_path = filepath + ".tmp"

        # Write to temp file
        write_fn(tmp_path)

        # Try to fsync to ensure bytes hit disk (best-effort)
        try:
            with open(tmp_path, "rb") as f:
                try:
                    os.fsync(f.fileno())
                except OSError:
                    # On some systems/drives this can fail; ignore durability-only failure
                    pass
        except FileNotFoundError:
            # If for some reason the temp file isn't there, let os.replace fail below
            pass

        # Atomic replace
        os.replace(tmp_path, filepath)

    # -------------------------------------------
    # CHECKPOINT FUNCTION
    # -------------------------------------------
    vectors_since_last_save = 0

    def save_checkpoint():
        nonlocal index, vectors_since_last_save

        if index is None:
            return

        tqdm.write("Checkpoint: saving index + mapping + state...")

        # 1. Index
        atomic_write_bytes(index_path, lambda tmp: faiss.write_index(index, tmp))

        # 2. Mapping
        def write_mapping(tmp):
            with open(tmp, "wb") as f:
                pickle.dump(idx_to_path, f, protocol=pickle.HIGHEST_PROTOCOL)

        atomic_write_bytes(mapping_path, write_mapping)

        # 3. State (now stores row-group counts)
        state_obj = {
            "per_file_groups_processed": per_file_groups_processed,
            "total_vectors": len(idx_to_path),
        }

        def write_state(tmp):
            with open(tmp, "w", encoding="utf-8") as f:
                json.dump(state_obj, f)

        atomic_write_bytes(state_path, write_state)

        vectors_since_last_save = 0
        tqdm.write("Checkpoint saved.")

    # -------------------------------------------
    # BATCH PARSING FUNCTION
    # -------------------------------------------
    def parse_embeddings_batch(batch: pa.RecordBatch, shard_name: str):
        """
        Returns:
            emb_matrix : np.ndarray | None, shape (n_valid, dim)
            paths      : list[str]
        """
        emb_field = batch["embeddings_result"]
        valid_mask = pc.is_valid(emb_field)
        if pc.sum(valid_mask).as_py() == 0:
            return None, []

        batch_filtered = batch.filter(valid_mask)
        emb_col = batch_filtered["embeddings_result"]
        idx_col = batch_filtered["original_image_index"]

        emb_type = emb_col.type

        try:
            # Fast path: fixed-size list of floats
            if isinstance(emb_type, pa.FixedSizeListType) and pa.types.is_floating(
                emb_type.value_type
            ):
                dim = emb_type.list_size
                flat = emb_col.values.to_numpy(zero_copy_only=False)
                if flat.shape[0] % dim != 0:
                    return None, []

                emb_matrix = flat.reshape(-1, dim).astype("float32", copy=False)
                idx_py = idx_col.to_pylist()
                paths = [f"{shard_name}_images/{i}.jpg" for i in idx_py]
                return emb_matrix, paths

            # Fallback: embeddings stored as lists/strings
            emb_py = emb_col.to_pylist()
            idx_py = idx_col.to_pylist()

            arrs, paths = [], []
            for e, i in zip(emb_py, idx_py):
                if e is None:
                    continue
                if isinstance(e, str):
                    try:
                        e = ast.literal_eval(e)
                    except Exception:
                        continue
                arrs.append(np.asarray(e, dtype="float32"))
                paths.append(f"{shard_name}_images/{i}.jpg")

            if not arrs:
                return None, []
            emb_matrix = np.vstack(arrs).astype("float32", copy=False)
            return emb_matrix, paths

        except Exception:
            return None, []

    # -------------------------------------------
    # MAIN PROCESSING LOOP (PER SHARD)
    # -------------------------------------------
    total_vectors_added_this_run = 0

    for shard_path in parquet_files:
        shard_path = str(shard_path)
        shard_name = Path(shard_path).stem

        # Open Parquet file and determine row groups
        pf = pq.ParquetFile(shard_path)
        num_row_groups = pf.metadata.num_row_groups

        # Number of row-groups already processed for this file (if any)
        groups_done = per_file_groups_processed.get(shard_path, 0)

        # Safety: if file changed (row-group count shrank), restart it
        if groups_done > num_row_groups:
            groups_done = 0

        # Row-group–based progress bar
        pbar = tqdm(
            total=num_row_groups,
            desc=f"Shard {shard_name}",
            unit="group",
            dynamic_ncols=True,
            leave=True,
        )

        # Move bar if resuming
        if groups_done > 0:
            pbar.update(groups_done)

        # Process remaining row groups
        for rg_idx in range(groups_done, num_row_groups):
            table = pf.read_row_group(rg_idx)
            batches = table.to_batches()

            for batch in batches:
                emb, paths = parse_embeddings_batch(batch, shard_name)

                if emb is not None and emb.shape[0] > 0:
                    emb = np.ascontiguousarray(emb, dtype="float32")
                    faiss.normalize_L2(emb)

                    if index is None:
                        dim = emb.shape[1]
                        index = faiss.IndexFlatIP(dim)

                    index.add(emb)
                    idx_to_path.extend(paths)
                    vectors_since_last_save += len(paths)
                    total_vectors_added_this_run += len(paths)

                    # Checkpoint if needed
                    if vectors_since_last_save >= save_every_vectors:
                        save_checkpoint()
                        gc.collect()

            # Mark this row group as completed
            per_file_groups_processed[shard_path] = rg_idx + 1

            # Update progress bar by 1 row group
            pbar.update(1)

        pbar.close()
        del pf
        gc.collect()

    # -------------------------------------------
    # FINAL SAVE
    # -------------------------------------------
    if index is None:
        tqdm.write("No embeddings found; nothing to save.")
        return

    save_checkpoint()

    tqdm.write("Index build complete.")
    tqdm.write(f"Total vectors before run: {total_vectors_existing}")
    tqdm.write(f"New vectors added: {total_vectors_added_this_run}")
    tqdm.write(f"Final total vectors: {len(idx_to_path)}")
    tqdm.write(f"Index saved → {index_path}")
    tqdm.write(f"Mapping saved → {mapping_path}")
    tqdm.write(f"State saved → {state_path}")


# =======================
# DRIVER CODE
# =======================

# List of embedding directories
dataset_dirs = [
    r"F:\Thesis\0000_embeddings_cleaned",
    r"F:\Thesis\0001_embeddings_cleaned",
    r"F:\Thesis\0002_embeddings_cleaned",
    r"F:\Thesis\0003_embeddings_cleaned",
]

output_base_dir = r"G:\Thesis\image_retrieval_faiss_indices"
os.makedirs(output_base_dir, exist_ok=True)

for dataset_dir in dataset_dirs:
    all_files = os.listdir(dataset_dir)
    parquet_files = [
        os.path.join(dataset_dir, f)
        for f in all_files
        if os.path.splitext(f)[1].lower() == ".parquet"
    ]

    dataset_base = Path(dataset_dir).stem

    for parquet_path in parquet_files:
        parquet_name = Path(parquet_path).stem

        # Dynamic output paths
        output_index_path = os.path.join(
            output_base_dir,
            f"faiss_{dataset_base}_{parquet_name}_IndexFlatIP.index",
        )
        output_mapping_path = os.path.join(
            output_base_dir,
            f"faiss_{dataset_base}_{parquet_name}_mapping.pkl",
        )
        output_state_path = os.path.join(
            output_base_dir,
            f"faiss_{dataset_base}_{parquet_name}_state.json",
        )

        print(f"Processing {dataset_base}/{parquet_name}...")

        build_faiss_index_with_mapping_resume_exact(
            parquet_files=[parquet_path],
            index_path=output_index_path,
            mapping_path=output_mapping_path,
            state_path=output_state_path,
            save_every_vectors=100_000,
        )

        print(f"Saved index to {output_index_path} and mapping to {output_mapping_path}\n")

Processing 0000_embeddings_cleaned/part-00000...


Shard part-00000:   0%|          | 0/10 [00:38<?, ?group/s]

Checkpoint: saving index + mapping + state...


Shard part-00000:  10%|█         | 1/10 [01:22<12:21, 82.35s/group]

Checkpoint saved.


...

Checkpoint saved.


Shard part-00008:  50%|█████     | 1/2 [04:07<03:18, 198.83s/group]

Checkpoint: saving index + mapping + state...


Shard part-00008: 100%|██████████| 2/2 [05:28<00:00, 164.11s/group]


Checkpoint saved.
Checkpoint: saving index + mapping + state...
Checkpoint saved.
Index build complete.
Total vectors before run: 0
New vectors added: 265287
Final total vectors: 265287
Index saved → G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_IndexFlatIP.index
Mapping saved → G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_mapping.pkl
State saved → G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_state.json
Saved index to G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_IndexFlatIP.index and mapping to G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_mapping.pkl



## Retrieve relevant images.

In [ ]:
import os
import json
import faiss
import heapq
import numpy as np
from transformers import AutoProcessor, AutoModel
import torch
import pickle
from pathlib import Path
from typing import List, Optional
from tqdm import tqdm

# ============================================================
# UTILITY: Chunk list
# ============================================================

def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


# ============================================================
# LOAD PROCESSED PROMPTS FROM JSONL
# ============================================================

def load_processed_prompts_from_jsonl(jsonl_path: str):
    processed = set()
    path = Path(jsonl_path)

    if not path.exists():
        return processed

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                p = obj.get("prompt")
                if p:
                    processed.add(p)
            except json.JSONDecodeError:
                continue

    print(f"[Resume] Found {len(processed)} previously processed prompts.")
    return processed


# ============================================================
# APPEND RESULTS TO JSONL
# ============================================================

def append_results_to_jsonl(jsonl_path: str, batch_results: list):
    path = Path(jsonl_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        for entry in batch_results:
            f.write(json.dumps(entry) + "\n")
            f.flush()
            os.fsync(f.fileno())

    print(f"[JSONL Append] {len(batch_results)} prompts appended → {jsonl_path}")


# ============================================================
# FAISS SEARCH FUNCTION (RELATIVE NEGATIVE DOMINANCE)
# ============================================================

def search_images(
    shard_index_paths: List[str],
    shard_mapping_paths: List[str],
    model_name: str,
    prompts: List[str],
    negative_prompts: Optional[List[str]] = None,
    top_k: int = 10,
    similarity_threshold: Optional[float] = None,
    dominance_margin: float = 0.0,   # <<< NEW (default = strict dominance)
    device: str = None
):
    """
    Batch FAISS search across shards.

    Rejects an image if:
        max(sim(image, negative_prompt)) >= sim(image, positive_prompt) - margin
    """

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # -------------------- Load CLIP --------------------
    processor = AutoProcessor.from_pretrained(model_name, force_download=False, use_fast=True)
    model = AutoModel.from_pretrained(model_name, force_download=False).to(device)
    model.eval()

    # -------------------- Encode positive prompts --------------------
    pos_inputs = processor(
        text=prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        pos_text_features = model.get_text_features(**pos_inputs)

    pos_text_features = pos_text_features / pos_text_features.norm(dim=-1, keepdim=True)
    pos_query_embs = pos_text_features.cpu().numpy().astype("float32")
    faiss.normalize_L2(pos_query_embs)

    n_queries = pos_query_embs.shape[0]
    all_results = [[] for _ in range(n_queries)]

    # Track seen embeddings per query (for deduplication)
    seen_embedding_hashes = [set() for _ in range(n_queries)]

    # -------------------- Encode negative prompts --------------------
    neg_embs = None
    if negative_prompts:
        neg_inputs = processor(
            text=negative_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            neg_features = model.get_text_features(**neg_inputs)

        neg_features = neg_features / neg_features.norm(dim=-1, keepdim=True)
        neg_embs = neg_features.cpu().numpy().astype("float32")

    # -------------------- Search shards --------------------
    for idx_path, map_path in tqdm(
        list(zip(shard_index_paths, shard_mapping_paths)),
        desc="Searching shards",
        unit="shard"
    ):
        idx_path = Path(idx_path)
        map_path = Path(map_path)

        shard_name = idx_path.stem
        parts = shard_name.split("_")
        group_id = parts[1] if len(parts) > 1 else "unknown"

        index = faiss.read_index(str(idx_path))
        with open(map_path, "rb") as f:
            idx_to_path = pickle.load(f)

        per_shard_k = min(len(idx_to_path), top_k) if top_k != None else len(idx_to_path)
        D, I = index.search(pos_query_embs, per_shard_k)

        for qi in range(n_queries):
            pos_query_emb = pos_query_embs[qi]

            for pos_sim, idx in zip(D[qi], I[qi]):
                idx = int(idx)
                if idx < 0 or idx >= len(idx_to_path):
                    continue
                if similarity_threshold is not None and pos_sim < similarity_threshold:
                    continue

                # Retrieve exact image embedding
                img_emb = index.reconstruct(idx)
                img_emb = img_emb / np.linalg.norm(img_emb)

                # ---------------- EMBEDDING DEDUPLICATION ----------------
                emb_hash = hash(img_emb.tobytes())

                # Skip if this embedding was already seen for this query
                if emb_hash in seen_embedding_hashes[qi]:
                    continue

                seen_embedding_hashes[qi].add(emb_hash)

                # ---------------- RELATIVE NEGATIVE DOMINANCE ----------------
                if neg_embs is not None:
                    neg_sim = np.max(img_emb @ neg_embs.T)

                    # Reject if negative concept dominates
                    if neg_sim >= (pos_sim - dominance_margin):
                        continue

                all_results[qi].append({
                    "image_path": idx_to_path[idx],
                    "score": float(pos_sim),
                    "shard": shard_name,
                    "group_id": group_id
                })

        del index

    # -------------------- Global Top-K --------------------
    final_results = []
    if top_k != None:
        for qi in range(n_queries):
            final_results.append({
                "prompt": prompts[qi],
                "results": heapq.nlargest(
                    top_k,
                    all_results[qi],
                    key=lambda x: x["score"]
                )
            })
    else:
        for qi in range(n_queries):
            final_results.append({
                "prompt": prompts[qi],
                "results": sorted(
                    all_results[qi],
                    key=lambda x: x["score"],
                    reverse=True
                )
            })

    return final_results


# ============================================================
# MAIN PROCESSING LOOP (WITH RESUME)
# ============================================================

def process_all_prompts_with_resume(
    shard_indexes,
    shard_mappings,
    profession_list,
    prompt_templates,
    jsonl_path,
    chunk_size,
    negative_prompts=[
        "logo",
        "company logo",
        "document",
        "text document",
        "paper",
        "printed text",
        "sign",
        "signboard"
    ],
    model_name="openai/clip-vit-large-patch14",
    similarity_threshold=0.15,
    dominance_margin=0.0,
    top_k=1000
):
    processed = load_processed_prompts_from_jsonl(jsonl_path)
    print(processed)

    all_prompts = []
    if prompt_templates != None:
        for p in sorted(set(profession_list)):
            for tmpl in prompt_templates:
                all_prompts.append(tmpl.format(object=p))
    else:
        all_prompts = profession_list
    
    print(all_prompts)

    for bidx, chunk in enumerate(chunk_list(all_prompts, chunk_size)):
        to_process = [p for p in chunk if p not in processed]

        if not to_process:
            print(f"[Batch {bidx}] All prompts already processed → Skipping.")
            continue

        print(f"[Batch {bidx}] Processing {len(to_process)} prompts: {to_process}")

        batch_results = search_images(
            shard_index_paths=shard_indexes,
            shard_mapping_paths=shard_mappings,
            model_name=model_name,
            prompts=to_process,
            negative_prompts=negative_prompts,
            top_k=top_k,
            similarity_threshold=similarity_threshold,
            dominance_margin=dominance_margin
        )

        append_results_to_jsonl(jsonl_path, batch_results)

        for r in batch_results:
            processed.add(r["prompt"])

    print("All batches processed.")

In [ ]:
from pathlib import Path
from tqdm import tqdm

faiss_dir = r"G:\Thesis\image_retrieval_faiss_indices"

shard_indexes = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_IndexFlatIP.index")])
shard_mappings = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_mapping.pkl")])
assert len(shard_indexes) == len(shard_mappings), "Mismatch between indexes and mappings!"

# List of professions retrieved from papers as well as https://www.surveycodings.org/articles/codings/occupation/download (occupations_ISCO08_5dgt_55languages_4000titles_with_mapping_surveycodings_20230425.xlsx) that all map to ISCO-08 for deriving demographics
profession_list = ['Accountant',
 'Actor',
 'Actuary',
 'Administrative Assistant',
 'Administrator',
 'Air Traffic Controller',
 'Animal Trainer',
 'Anthropologist',
 'Appraiser',
 'Archaeologist',
 'Architect',
 'Archivist',
 'Art Director',
 'Artist',
 'Astronaut',
 'Astronomer',
 'Athlete',
 'Audio Technician',
 'Auditor',
 'Automotive Designer',
 'Baker',
 'Banker',
 'Bankruptcy Specialist',
 'Barber',
 'Barista',
 'Bartender',
 'Basketball player',
 'Biologist',
 'Biomedical Engineer',
 'Blacksmith',
 'Bodyguard',
 'Bounty Hunter',
 'Boxer',
 'Brand Manager',
 'Brewer',
 'Bricklayer',
 'Broker',
 'Builder',
 'Butcher',
 'CEO',
 'Carer',
 'Carpenter',
 'Cartographer',
 'Cashier',
 'Chef',
 'Chemical Engineer',
 'Chemist',
 'Chiropractor',
 'Civil Engineer',
 'Claims Adjuster',
 'Cleaner',
 'Clerk',
 'Coach',
 'Comedian',
 'Compliance Officer',
 'Composer',
 'Conservation Officer',
 'Construction Worker',
 'Copywriter',
 'Court Reporter',
 'Crime Scene Investigator',
 'Customer Support Specialist',
 'DJ',
 'Dancer',
 'Data Scientist',
 'Database Administrator',
 'Debt Counselor',
 'Dentist',
 'Detective',
 'Development Officer',
 'Dietitian',
 'Director',
 'Doctor',
 'Dog Walker',
 'Draughtsperson',
 'Driver',
 'Economist',
 'Editor',
 'Electrician',
 'Emergency Management Specialist',
 'Entrepreneur',
 'Environmental Engineer',
 'Ergonomist',
 'Estate Planner',
 'Event Coordinator',
 'Executive Assistant',
 'Exterminator',
 'Facilities Manager',
 'Farmer',
 'Fashion Designer',
 'Firefighter',
 'Fishmonger',
 'Flight Attendant',
 'Florist',
 'Football player',
 'Forklift Operator',
 'Gardener',
 'Geologist',
 'Graphic Designer',
 'Grocer',
 'Hair dresser',
 'Handyperson',
 'Health Inspector',
 'Historian',
 'Hotel Concierge',
 'Hotel Manager',
 'Human Resources Specialist',
 'IT Support Specialist',
 'Illustrator',
 'Industrial Designer',
 'Insurance Underwriter',
 'Janitor',
 'Jeweller',
 'Journalist',
 'Judge',
 'Lawyer',
 'Librarian',
 'Lifeguard',
 'Loan Officer',
 'Logger',
 'Logistics Manager',
 'Magician',
 'Makeup Artist',
 'Marine Biologist',
 'Marketing Manager',
 'Masseur',
 'Mathematician',
 'Mayor',
 'Mechanic',
 'Meteorologist',
 'Midwife',
 'Miner',
 'Model',
 'Musician',
 'News Reader',
 'Nurse',
 'Nutritionist',
 'Oceanographer',
 'Office Assistant',
 'Operations Manager',
 'Optician',
 'Painter',
 'Paralegal',
 'Paramedic',
 'Park Ranger',
 'Payroll Specialist',
 'Personal Trainer',
 'Pharmacist',
 'Photographer',
 'Physicist',
 'Pilot',
 'Plumber',
 'Police Officer',
 'Politician',
 'Postal Worker',
 'Priest',
 'Procurement Officer',
 'Professor',
 'Property Manager',
 'Psychologist',
 'Quality Assurance Inspector',
 'Real Estate Agent',
 'Receptionist',
 'Researcher',
 'Roofer',
 'Safety Inspector',
 'Sailor',
 'Salesperson',
 'Scientist',
 'Security Officer',
 'Shopkeeper',
 'Singer',
 'Skier',
 'Social Worker',
 'Software Engineer',
 'Soldier',
 'Sound Engineer',
 'Statistician',
 'Street Vendor',
 'Surfer',
 'Surgeon',
 'Swimmer',
 'Tailor',
 'Tattoo Artist',
 'Teacher',
 'Technician',
 'Tennis Player',
 'Therapist',
 'Translator',
 'Umpire',
 'Urban Planner',
 'Usher',
 'Veterinarian',
 'Videographer',
 'Waiter',
 'Waste Collection Worker',
 'Welder',
 'Wholesaler',
 'Writer',
 'Zoologist']

# prompt_templates = ["Male {object}", "Female {object}"]
prompt_templates = None # Gender invariant prompts are being created below via make_prompt to assign correct articles

def choose_article(noun: str) -> str:
    """
    Returns 'a' or 'an' based on the first letter of the noun.
    Sufficient for occupation names used in CLIP prompts.
    """
    return "an" if noun[0].lower() in "aeiou" else "a"

def make_prompt(profession: str) -> str:
    article = choose_article(profession)
    return f"A photo of {article} {profession.lower()}"


profession_list = [make_prompt(p) for p in profession_list]

jsonl_path = r"E:\ImageRetrieval\Professions_125k_ISCO_Aligned\ISCO_aligned_125k_retrieval_results_batchsize_10.jsonl"

process_all_prompts_with_resume(
    shard_indexes=shard_indexes,
    shard_mappings=shard_mappings,
    profession_list=profession_list,
    prompt_templates=prompt_templates,
    jsonl_path=jsonl_path,
    chunk_size=10,
    negative_prompts = ["Cartoon", "NSFW", "Sex", "Naked", "Clothing", "Object", "Sign", "Logo", "Document", "Paper", "Page", "Animal", "Cat", "Dog"],
    model_name="openai/clip-vit-large-patch14",
    similarity_threshold=0.15,
    dominance_margin=0.01,
    top_k=125_000 #None #200_000
)

set()
['A photo of an accountant', 'A photo of an actor', 'A photo of an actuary', 'A photo of an administrative assistant', 'A photo of an administrator', 'A photo of an air traffic controller', 'A photo of an animal trainer', 'A photo of an anthropologist', 'A photo of an appraiser', 'A photo of an archaeologist', 'A photo of an architect', 'A photo of an archivist', 'A photo of an art director', 'A photo of an artist', 'A photo of an astronaut', 'A photo of an astronomer', 'A photo of an athlete', 'A photo of an audio technician', 'A photo of an auditor', 'A photo of an automotive designer', 'A photo of a baker', 'A photo of a banker', 'A photo of a bankruptcy specialist', 'A photo of a barber', 'A photo of a barista', 'A photo of a bartender', 'A photo of a basketball player', 'A photo of a biologist', 'A photo of a biomedical engineer', 'A photo of a blacksmith', 'A photo of a bodyguard', 'A photo of a bounty hunter', 'A photo of a boxer', 'A photo of a brand manager', 'A photo of

c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Searching shards: 100%|██████████| 36/36 [23:02<00:00, 38.40s/shard]


[JSONL Append] 10 prompts appended → G:\Thesis\ImageRetrieval\Professions_125k_ISCO_Aligned\retrieval_results_batchsize_10.jsonl
[Batch 1] Processing 10 prompts: ['A photo of an architect', 'A photo of an archivist', 'A photo of an art director', 'A photo of an artist', 'A photo of an astronaut', 'A photo of an astronomer', 'A photo of an athlete', 'A photo of an audio technician', 'A photo of an auditor', 'A photo of an automotive designer']


c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Searching shards: 100%|██████████| 36/36 [23:00<00:00, 38.36s/shard]


[JSONL Append] 10 prompts appended → G:\Thesis\ImageRetrieval\Professions_125k_ISCO_Aligned\retrieval_results_batchsize_10.jsonl
[Batch 2] Processing 10 prompts: ['A photo of a baker', 'A photo of a banker', 'A photo of a bankruptcy specialist', 'A photo of a barber', 'A photo of a barista', 'A photo of a bartender', 'A photo of a basketball player', 'A photo of a biologist', 'A photo of a biomedical engineer', 'A photo of a blacksmith']


...

[JSONL Append] 10 prompts appended → G:\Thesis\ImageRetrieval\Professions_125k_ISCO_Aligned\retrieval_results_batchsize_10.jsonl
[Batch 18] Processing 10 prompts: ['A photo of a surgeon', 'A photo of a swimmer', 'A photo of a tailor', 'A photo of a tattoo artist', 'A photo of a teacher', 'A photo of a technician', 'A photo of a tennis player', 'A photo of a therapist', 'A photo of a translator', 'A photo of an umpire']


Searching shards: 100%|██████████| 36/36 [22:02<00:00, 36.73s/shard]


[JSONL Append] 10 prompts appended → G:\Thesis\ImageRetrieval\Professions_125k_ISCO_Aligned\retrieval_results_batchsize_10.jsonl
[Batch 19] Processing 10 prompts: ['A photo of an urban planner', 'A photo of an usher', 'A photo of a veterinarian', 'A photo of a videographer', 'A photo of a waiter', 'A photo of a waste collection worker', 'A photo of a welder', 'A photo of a wholesaler', 'A photo of a writer', 'A photo of a zoologist']


Searching shards: 100%|██████████| 36/36 [20:45<00:00, 34.61s/shard]


[JSONL Append] 10 prompts appended → G:\Thesis\ImageRetrieval\Professions_125k_ISCO_Aligned\retrieval_results_batchsize_10.jsonl
All batches processed.
